In [23]:
import numpy as np
import pandas as pd
from pathlib import Path

import anndata as ad
import scanpy as sc
from scipy import sparse

from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold

import torch
import torch.nn as nn

from myllia_metric import myllia_score

SEED = 6
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(DEVICE)

EMB_DIM_PERT = 128   # pert gene embedding dim (from SVD)
EMB_DIM_OUT = 128   # output gene embedding dim (from SVD)
RANK_R = 32    # low-rank interaction size

DROPOUT = 0.10
LR = 2e-3 * 0.6
WD = 1e-4
EPOCHS = 400
BATCH_GENES = 16 # minibatch over perturbed genes
EVAL_EVERY = 25
PATIENCE = 12 # early stopping patience in eval steps

# Gate parameters should match metric structure
GATE_A = 0.0
GATE_B = 0.2
EPS = 1e-12

ROOT = Path(".")

# --- added: CV alpha sweep + refit ---
MODEL_SEEDS = [6, 7, 8]
ALPHA_GRID = [0.7, 0.75, 0.778, 0.82, 0.86]
GRAD_CLIP = 1.0
H5AD_PATH = ROOT / "data" / "training_cells.h5ad"  # used ONLY for perts not in gene_columns


In [24]:
def score_delta(dt, dp):
    dt = dt.astype(np.float32, copy=False)
    dp = dp.astype(np.float32, copy=False)
    r = myllia_score(dt, dp)
    return {
        "score": float(r.score),
        "wcos": float(r.wcos),
        "mean_term": float(r.mean_term),
        "pred_wmae": float(r.pred_wmae),
    }


In [25]:
means_path = ROOT / "data" / "training_data_means.csv"
valmap_path = ROOT / "data" / "pert_ids_val.csv"
sample_sub_path = ROOT / "data" / "sample_submission.csv"

df_means = pd.read_csv(means_path)
df_valmap = pd.read_csv(valmap_path)
df_sub = pd.read_csv(sample_sub_path)

gene_columns = [c for c in df_means.columns if c != "pert_symbol"]

baseline_mask = df_means["pert_symbol"].astype(str) == "non-targeting"
x_base = df_means.loc[baseline_mask, gene_columns].iloc[0].to_numpy(np.float32)

df_train = df_means.loc[~baseline_mask].reset_index(drop=True)
train_genes = df_train["pert_symbol"].astype(str).to_numpy()

X_train_means = df_train[gene_columns].to_numpy(np.float32)
D_train = X_train_means - x_base[None, :] # (80, 5127) delta vs non-targeting

delta_baseline = D_train.mean(axis=0).astype(np.float32)

# pert_id -> gene symbol for leaderboard (first 60)
val_map = dict(zip(df_valmap["pert_id"].astype(str), df_valmap["pert"].astype(str)))

print("Train perts:", len(train_genes), "G:", len(gene_columns))
print("Sample submission rows:", len(df_sub))
print("Val mapping entries:", len(val_map))


Train perts: 80 G: 5127
Sample submission rows: 120
Val mapping entries: 60


Build gene embeddings from D_train (SVD on signed-log transformed deltas). Build embeddings for output genes (gene_columns) via SVD on (80, 5127). Pert embeddings are looked up by gene symbol in gene_columns, otherwise fallback to the mean embedding.

In [ ]:
val_targets = df_valmap["pert"].astype(str).tolist()

union_genes = sorted(set([g.upper() for g in gene_columns] +
                         [g.upper() for g in train_genes.tolist()] +
                         [g.upper() for g in val_targets]))

geneU = pd.Index([str(g).upper() for g in gene_columns])

# (80, 5127) dense -> signed log transform to handle negatives
X = D_train.astype(np.float32, copy=True)
X = np.sign(X) * np.log2(1.0 + np.abs(X))

# SVD across genes: components_ is (k, 5127) so transpose to (5127, k)
svd = TruncatedSVD(n_components=max(EMB_DIM_PERT, EMB_DIM_OUT), random_state=SEED)
svd.fit(X)

gene_emb_all = svd.components_.T.astype(np.float32)  # (G, k)
k_svd = gene_emb_all.shape[1]
print("SVD k:", k_svd, "gene_emb_all:", gene_emb_all.shape)

# Map output genes to embeddings
gene2emb_pert = {gene_columns[i].upper(): gene_emb_all[i, :EMB_DIM_PERT].copy()
                 for i in range(len(gene_columns))}
gene2emb_out  = {gene_columns[i].upper(): gene_emb_all[i, :EMB_DIM_OUT ].copy()
                 for i in range(len(gene_columns))}

emb_fallback_pert = gene_emb_all[:, :EMB_DIM_PERT].mean(axis=0).astype(np.float32)
emb_fallback_out  = gene_emb_all[:, :EMB_DIM_OUT ].mean(axis=0).astype(np.float32)

missing_emb_pert = {}

def emb_pert(g: str) -> np.ndarray:
    gU = str(g).upper()
    if gU in gene2emb_pert:
        return gene2emb_pert[gU]
    if gU in missing_emb_pert:
        return missing_emb_pert[gU]
    return emb_fallback_pert

def emb_out(g: str) -> np.ndarray:
    return gene2emb_out.get(str(g).upper(), emb_fallback_out)

# Output gene embeddings in the exact output gene order
U_out = np.vstack([emb_out(g) for g in gene_columns]).astype(np.float32)    # (G, d_out)
Z_train = np.vstack([emb_pert(g) for g in train_genes]).astype(np.float32)  # (80, d_pert)

print("U_out:", U_out.shape, "Z_train:", Z_train.shape)

# Optional: visibility into coverage
missing_train = [g for g in train_genes.tolist() if str(g).upper() not in geneU]
missing_val = [g for g in val_targets if str(g).upper() not in geneU]
if missing_train:
    print(f"train perts not in gene_columns: {len(missing_train)}. Example: {missing_train[:12]}")
if missing_val:
    print(f"val perts not in gene_columns: {len(missing_val)}. Example: {missing_val[:12]}")


# h5ad for missing perts: embed by control-cell coexpression
missing_all = sorted(set([str(x).upper() for x in (missing_train + missing_val)]))

if len(missing_all) > 0:
    try:
        import scipy.sparse as sp
        import anndata as ad

        print("[h5ad] building embeddings for missing perts:", len(missing_all))
        adata = ad.read_h5ad(str(H5AD_PATH))

        # pick perturbation column
        pert_col = None
        for c in ["sgrna_symbol", "pert_symbol", "pert", "perturbation", "gene", "target_gene"]:
            if c in adata.obs.columns:
                pert_col = c
                break
        if pert_col is None:
            raise ValueError("Could not find perturbation column in h5ad obs.")

        Xc = adata.X
        if not sp.issparse(Xc):
            Xc = sp.csr_matrix(Xc)
        else:
            Xc = Xc.tocsr()

        # normalize ALL genes: CPM10K then log2(1+x)
        cell_sum = np.asarray(Xc.sum(axis=1)).ravel().astype(np.float64)
        scale = (10000.0 / cell_sum).astype(np.float64)

        Xn = Xc.multiply(scale[:, None]).tocsr()
        Xn.data = np.log1p(Xn.data) / np.log(2.0)

        ctrl_mask = (adata.obs[pert_col].astype(str).to_numpy() == "non-targeting")
        if int(ctrl_mask.sum()) == 0:
            raise ValueError("No non-targeting control cells found in h5ad.")

        var = {str(g).upper(): i for i, g in enumerate(adata.var_names.astype(str).to_numpy())}

        # indices for the 5127 output genes in the 19,226 gene space
        out_idx = np.array([var[str(g).upper()] for g in gene_columns if str(g).upper() in var], dtype=np.int64)
        if len(out_idx) != len(gene_columns):
            miss_out = [g for g in gene_columns if str(g).upper() not in var]
            raise ValueError(f"{len(miss_out)} output genes missing from h5ad var_names (unexpected). Example: {miss_out[:10]}")

        Xout = Xn[ctrl_mask][:, out_idx]  # (n_ctrl, 5127)
        if sp.issparse(Xout):
            Xout = Xout.toarray()
        Xout = Xout.astype(np.float32)

        mu = Xout.mean(axis=0, keepdims=True)
        sd = Xout.std(axis=0, keepdims=True) + 1e-6
        Xout_z = (Xout - mu) / sd

        # use the *existing* pert embedding space (from SVD on D_train): (5127, d_pert)
        P_out = gene_emb_all[:, :EMB_DIM_PERT].astype(np.float32)

        topk = 256
        made = 0
        for gU in missing_all:
            if gU not in var:
                continue

            xg = Xn[ctrl_mask, var[gU]]
            if sp.issparse(xg):
                xg = xg.toarray()
            xg = np.asarray(xg).ravel().astype(np.float32)

            xg = (xg - xg.mean()) / (xg.std() + 1e-6)

            corr = (xg[:, None] * Xout_z).mean(axis=0)  # (5127,)
            idx = np.argsort(-np.abs(corr))[:topk]
            w = corr[idx].astype(np.float32)

            z = (w[:, None] * P_out[idx]).sum(axis=0)
            z = z / (np.linalg.norm(z) + 1e-12)

            missing_emb_pert[gU] = z.astype(np.float32)
            made += 1

        print("[h5ad] embedded missing perts:", made, "of", len(missing_all))
    except Exception as e:
        print("[h5ad] skipped missing pert embeddings due to error:", repr(e))


SVD k: 80 gene_emb_all: (5127, 80)
U_out: (5127, 80) Z_train: (80, 80)
train perts not in gene_columns: 8. Example: ['BRD4', 'CHD4', 'DNAJA3', 'INO80', 'KAT8', 'KDM4A', 'PMEL', 'SETD1A']
val perts not in gene_columns: 8. Example: ['SMARCB1', 'PSMA1', 'CUL1', 'FLT4', 'FOXH1', 'HK2', 'TRAM2', 'DPH2']
[h5ad] building embeddings for missing perts: 16
[h5ad] embedded missing perts: 16 of 16


In [ ]:
def gate_smoothstep(x, a = GATE_A, b = GATE_B):
    t = (x - a) / (b - a)
    t = torch.clamp(t, 0.0, 1.0)
    return t * t * (3.0 - 2.0 * t)

def per_row_weighted_l1_like(delta_true: torch.Tensor, delta_pred: torch.Tensor, eps: float = EPS) -> torch.Tensor:
    """
    Same logic as weighted_l1_like, but returns (N,) per-row.
    """
    w = gate_smoothstep(torch.abs(delta_true), a=GATE_A, b=GATE_B)  # (N, G)
    err = torch.abs(delta_pred - delta_true)                        # (N, G)
    num = torch.sum(w * err, dim=1)                                 # (N,)
    den = torch.clamp(torch.sum(w, dim=1), min=eps)                 # (N,)
    return num / den                                                # (N,)

import torch

def weighted_l1_like_rowweighted(
    delta_true: torch.Tensor,     # (N, G)
    delta_pred: torch.Tensor,     # (N, G)
    baseline_wmae: torch.Tensor,  # (N,)
    *,
    eps: float = 1e-8,
    mode: str = "inv_sqrt",       # "inv", "inv_sqrt", "inv_log"
    clamp_min: float = 0.5,
    clamp_max: float = 3.0,
) -> torch.Tensor:
    """
    Weight idea:
      - baseline_wmae small => ratio metric is unforgiving => upweight that row
      - baseline_wmae large => easier => downweight a bit

    Returns: scalar loss
    """
    per_row = per_row_weighted_l1_like(delta_true, delta_pred, eps=eps)  # (N,)

    b = baseline_wmae.to(delta_true.device).to(delta_true.dtype)

    if mode == "inv":
        w = 1.0 / (b + eps)
    elif mode == "inv_sqrt":
        w = 1.0 / torch.sqrt(b + eps)
    elif mode == "inv_log":
        w = 1.0 / torch.log1p(b + eps)
    else:
        raise ValueError(f"Unknown mode={mode}")

    # clamp to prevent a few rows from dominating training
    w = torch.clamp(w, min=clamp_min, max=clamp_max)

    # normalized weighted mean (stable)
    return torch.sum(w * per_row) / torch.clamp(torch.sum(w), min=eps)

def weighted_cosine_per_row_torch(dt: torch.Tensor, dp: torch.Tensor, w: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    # dt, dp, w: (B, G)
    wa = w * dt
    wb = w * dp
    num = torch.sum(wa * wb, dim=1)
    da  = torch.sqrt(torch.sum(wa * wa, dim=1))
    db  = torch.sqrt(torch.sum(wb * wb, dim=1))
    denom = torch.clamp(da * db, min=eps)
    return num / denom  # (B,)


In [28]:
class BilinearDeltaModel(nn.Module):
    def __init__(self, d_pert, d_out, rank_r, dropout):
        super().__init__()
        self.rank_r = rank_r

        self.proj_p = nn.Sequential(
            nn.Linear(d_pert, rank_r),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.proj_o = nn.Sequential(
            nn.Linear(d_out, rank_r),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # biases
        self.bias_global = nn.Parameter(torch.zeros(1))
        self.bias_gene = None  # set via set_gene_bias

    def set_gene_bias(self, G):
        dev = next(self.parameters()).device
        self.bias_gene = torch.nn.Parameter(torch.zeros(G, device=dev))
        self.bias_global = torch.nn.Parameter(torch.zeros(1, device=dev))

    def forward(self, z_pert, u_out):
        p = self.proj_p(z_pert)    # (B, R)
        o = self.proj_o(u_out)     # (G, R)
        y = p @ o.T                # (B, G)
        y = y + self.bias_gene[None, :] + self.bias_global
        return y

In [29]:
Y = D_train.astype(np.float32)
G = Y.shape[1]
N = Y.shape[0]

Uo_t = torch.tensor(U_out, device=device) # (G, d_out)
Zt = torch.tensor(Z_train, device=device) # (N, d_pert)
Yt = torch.tensor(Y, device=device) # (N, G)

print("N:", N, "G:", G, "device:", device)

N: 80 G: 5127 device: cuda


In [30]:
gt_df = pd.read_csv('Data/training_data_ground_truth_table.csv')
baseline_wmae_t = torch.tensor(
    gt_df["baseline_wmae"].to_numpy(dtype=np.float32),
    device=device,
    dtype=torch.float32,
)

In [31]:

def apply_shrink(pred, baseline, alpha):
    # pred: (B,G) ; baseline: (G,)
    return float(alpha) * pred + (1.0 - float(alpha)) * baseline[None, :]

def train_one_fold(tr_idx, va_idx, seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = BilinearDeltaModel(
        d_pert=Zt.shape[1],
        d_out=Uo_t.shape[1],
        rank_r=RANK_R,
        dropout=DROPOUT
    ).to(device)

    model.set_gene_bias(G)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    tr_idx = np.asarray(tr_idx)
    va_idx = np.asarray(va_idx)
    va_idx_t = torch.tensor(va_idx, device=device, dtype=torch.long)

    best_score = -1e18
    best_alpha = 0.0
    best_epoch = 0
    best_state = None
    best_va_pred = None  # unshrunk predictions at best checkpoint
    patience = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = tr_idx.copy()
        np.random.shuffle(perm)

        for start in range(0, len(perm), BATCH_GENES):
            b = perm[start:start + BATCH_GENES]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(Zt.index_select(0, b_t), Uo_t)
            dt_b = Yt.index_select(0, b_t)                 # (B, G)
            bw_b = baseline_wmae_t.index_select(0, b_t)     # (B,)

            loss = weighted_l1_like_rowweighted(
                dt_b, pred, bw_b,
                mode="inv_sqrt",
                clamp_min=0.5,
                clamp_max=3.0,
            )

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

        if epoch % 5 == 0 or epoch == EPOCHS:
            model.eval()
            with torch.no_grad():
                va_pred = model(Zt.index_select(0, va_idx_t), Uo_t).detach().cpu().numpy().astype(np.float32)

            va_true = Y[va_idx]

            # alpha sweep (0..0.7) on this fold
            sc_best = -1e18
            a_best = 0.0
            for a in ALPHA_GRID:
                pred_a = apply_shrink(va_pred, delta_baseline, float(a))
                sc = score_delta(va_true, pred_a)["score"]
                if sc > sc_best:
                    sc_best = sc
                    a_best = float(a)

            if sc_best > best_score:
                best_score = sc_best
                best_alpha = a_best
                best_epoch = epoch
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                best_va_pred = va_pred.copy()
                patience = 0
            else:
                patience += 1

            if patience >= PATIENCE:
                break

    return best_score, best_alpha, best_epoch, best_state, best_va_pred

# --- CV: collect OOF preds and optimize a GLOBAL alpha on OOF only ---
kf = KFold(n_splits=8, shuffle=True, random_state=SEED)

oof_pred = np.zeros_like(Y, dtype=np.float32)
oof_hit = np.zeros((N,), dtype=np.int32)

fold_scores = []
fold_alphas = []
fold_epochs = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(np.arange(N)), 1):
    best_score, best_alpha, best_epoch, best_state, best_va_pred = train_one_fold(tr_idx, va_idx, seed=SEED)
    fold_scores.append(float(best_score))
    fold_alphas.append(float(best_alpha))
    fold_epochs.append(int(best_epoch))

    oof_pred[va_idx] = best_va_pred
    oof_hit[va_idx] += 1

    print(f"fold {fold}: best_score={best_score:.6f} best_alpha={best_alpha:.3f} best_epoch={best_epoch}")

if not np.all(oof_hit == 1):
    print("[warn] OOF coverage not 1 everywhere. min/max:", int(oof_hit.min()), int(oof_hit.max()))

print("cv mean:", float(np.mean(fold_scores)), "std:", float(np.std(fold_scores)))

EPOCHS_MED = int(np.median(fold_epochs))
print("median best_epoch =", EPOCHS_MED)

# Global alpha chosen on OOF predictions only (more stable than per-fold alpha roulette)
best_global_alpha = 0.0
best_global_score = -1e18
for a in ALPHA_GRID:
    pred_a = apply_shrink(oof_pred, delta_baseline, float(a))
    sc = score_delta(Y, pred_a)["score"]
    if sc > best_global_score:
        best_global_score = sc
        best_global_alpha = float(a)

print("OOF global alpha:", best_global_alpha, "OOF score:", best_global_score)

ALPHA_SHRINK = best_global_alpha


fold 1: best_score=0.166475 best_alpha=0.860 best_epoch=25
fold 2: best_score=0.115051 best_alpha=0.778 best_epoch=45
fold 3: best_score=0.114235 best_alpha=0.700 best_epoch=25
fold 4: best_score=0.122564 best_alpha=0.860 best_epoch=35
fold 5: best_score=0.159363 best_alpha=0.860 best_epoch=25
fold 6: best_score=0.191482 best_alpha=0.700 best_epoch=30
fold 7: best_score=0.181774 best_alpha=0.750 best_epoch=20
fold 8: best_score=0.128870 best_alpha=0.700 best_epoch=20
cv mean: 0.14747667565431655 std: 0.029022193423455175
median best_epoch = 25
OOF global alpha: 0.778 OOF score: 0.1457519906800826


fold 1: best_score=0.166475 best_alpha=0.860 best_epoch=25
fold 2: best_score=0.115051 best_alpha=0.778 best_epoch=45
fold 3: best_score=0.114235 best_alpha=0.700 best_epoch=25
fold 4: best_score=0.122564 best_alpha=0.860 best_epoch=35
fold 5: best_score=0.159363 best_alpha=0.860 best_epoch=25
fold 6: best_score=0.191482 best_alpha=0.700 best_epoch=30
fold 7: best_score=0.181774 best_alpha=0.750 best_epoch=20
fold 8: best_score=0.128870 best_alpha=0.700 best_epoch=20
cv mean: 0.14747667565431655 std: 0.029022193423455175
median best_epoch = 25
OOF global alpha: 0.778 OOF score: 0.1457519906800826

In [10]:

def fit_full_model(seed, epochs_fixed):
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = BilinearDeltaModel(
        d_pert=Zt.shape[1],
        d_out=Uo_t.shape[1],
        rank_r=RANK_R,
        dropout=DROPOUT
    ).to(device)

    model.set_gene_bias(G)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    for epoch in range(1, epochs_fixed + 1):
        model.train()
        perm = np.arange(N)
        np.random.shuffle(perm)

        for start in range(0, N, BATCH_GENES):
            b = perm[start:start + BATCH_GENES]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(Zt.index_select(0, b_t), Uo_t)
            dt_b = Yt.index_select(0, b_t)                 # (B, G)
            bw_b = baseline_wmae_t.index_select(0, b_t)     # (B,)

            loss = weighted_l1_like_rowweighted(
                dt_b, pred, bw_b,
                mode="inv_sqrt",
                clamp_min=0.5,
                clamp_max=3.0,
            )

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

        if epoch % EVAL_EVERY == 0 or epoch == epochs_fixed:
            model.eval()
            with torch.no_grad():
                pred_np = model(Zt, Uo_t).detach().cpu().numpy().astype(np.float32)

            pred_np = apply_shrink(pred_np, delta_baseline, ALPHA_SHRINK)
            s = score_delta(Y, pred_np)
            print(f"[seed {seed}] epoch={epoch:4d} train_score={s['score']:.6f} wcos={s['wcos']:.6f} pred_wmae={s['pred_wmae']:.6f} alpha={ALPHA_SHRINK:.3f}")

    model.eval()
    return model

# refit ensemble on ALL 80 perts, using CV-calibrated epoch + OOF-calibrated alpha
models = [fit_full_model(sd, epochs_fixed=EPOCHS_MED) for sd in MODEL_SEEDS]
print("Refit models:", len(models))

def predict_delta_gene(gene_symbol: str) -> np.ndarray:
    z = torch.tensor(emb_pert(gene_symbol)[None, :].astype(np.float32), device=device)
    preds = []
    with torch.no_grad():
        for m in models:
            y = m(z, Uo_t).detach().cpu().numpy().astype(np.float32)[0]
            preds.append(y)
    yhat = np.mean(np.stack(preds, axis=0), axis=0).astype(np.float32)
    yhat = (ALPHA_SHRINK * yhat + (1.0 - ALPHA_SHRINK) * delta_baseline).astype(np.float32)
    return yhat

# Build submission from sample_submission.csv
sub = df_sub.copy()
sub["pert_id"] = sub["pert_id"].astype(str)
sub_gene_cols = [c for c in sub.columns if c != "pert_id"]

# ensure order matches gene_columns
idx = {g: i for i, g in enumerate(gene_columns)}
perm = [idx[g] for g in sub_gene_cols]

# fill default baseline for unknown test perts
sub.loc[:, sub_gene_cols] = np.tile(delta_baseline[perm][None, :], (len(sub), 1))

# fill known val perts (pert_1..pert_60)
hit = 0
for pid, gene in val_map.items():
    vec = predict_delta_gene(gene)[perm]
    m = (sub["pert_id"] == str(pid))
    if m.any():
        sub.loc[m, sub_gene_cols] = vec[None, :]
        hit += int(m.sum())

out_path = "submission_bilinear_refit_oofalpha.csv"
sub.to_csv(out_path, index=False)
print("[ok] wrote:", out_path, "| filled:", hit)


[seed 6] epoch=  25 train_score=0.191766 wcos=0.490485 pred_wmae=0.072424 alpha=0.778
[seed 7] epoch=  25 train_score=0.192180 wcos=0.490856 pred_wmae=0.072399 alpha=0.778
[seed 8] epoch=  25 train_score=0.191822 wcos=0.490592 pred_wmae=0.072431 alpha=0.778
Refit models: 3
[ok] wrote: submission_bilinear_refit_oofalpha.csv | filled: 60


[seed 6] epoch=  25 train_score=0.190763 wcos=0.493199 pred_wmae=0.072467 alpha=0.700
[seed 7] epoch=  25 train_score=0.191113 wcos=0.493597 pred_wmae=0.072458 alpha=0.700
[seed 8] epoch=  25 train_score=0.190862 wcos=0.493393 pred_wmae=0.072469 alpha=0.700
Refit models: 3
[ok] wrote: submission_bilinear_refit_oofalpha.csv | filled: 60

epoch=  25 train_score=0.197565 wcos=0.487435 pred_wmae=0.071318
epoch=  50 train_score=0.214566 wcos=0.506001 pred_wmae=0.070564
epoch=  75 train_score=0.255586 wcos=0.546834 pred_wmae=0.068880
epoch= 100 train_score=0.320180 wcos=0.596986 pred_wmae=0.066354
epoch= 125 train_score=0.395301 wcos=0.641472 pred_wmae=0.063575
epoch= 150 train_score=0.465017 wcos=0.673950 pred_wmae=0.061027
epoch= 175 train_score=0.528761 wcos=0.699482 pred_wmae=0.058800
epoch= 200 train_score=0.583847 wcos=0.718520 pred_wmae=0.056899
epoch= 225 train_score=0.634970 wcos=0.734100 pred_wmae=0.055180
epoch= 250 train_score=0.678502 wcos=0.748007 pred_wmae=0.053726
epoch= 275 train_score=0.714378 wcos=0.758516 pred_wmae=0.052467
epoch= 300 train_score=0.750164 wcos=0.767519 pred_wmae=0.051263
epoch= 325 train_score=0.780506 wcos=0.776008 pred_wmae=0.050237
epoch= 350 train_score=0.811266 wcos=0.784896 pred_wmae=0.049284
epoch= 375 train_score=0.837425 wcos=0.791671 pred_wmae=0.048401
epoch= 400 train_score=0.859289 wcos=0.797939 pred_wmae=0.047662
wrote: submission_bilinear.csv